# Cyclone Forecasting Stage-1 Replication on Kaggle (Exact Logic)

This notebook is for a **fresh Kaggle notebook** with dataset attached at:
- `/kaggle/input/final-dataset-with-splits/dataloaders_fixed`
- `/kaggle/input/final-dataset-with-splits/split_dataloaders/train_loader.pkl`
- `/kaggle/input/final-dataset-with-splits/split_dataloaders/test_loader.pkl`

Goal:
- Run original author Stage-1 scripts in Kaggle with infrastructure/path adaptation only.
- No training or model logic changes.


## Step 0: Define Inputs and Validate Them

Run this cell first.
Expected result:
- It prints `Input validation passed.`


In [ ]:
from pathlib import Path

DATASET_ROOT = Path('/kaggle/input/final-dataset-with-splits')
DATALOADER_SRC = DATASET_ROOT / 'dataloaders_fixed'
TRAIN_SPLIT_PKL = DATASET_ROOT / 'split_dataloaders' / 'train_loader.pkl'
TEST_SPLIT_PKL = DATASET_ROOT / 'split_dataloaders' / 'test_loader.pkl'

REPO_URL = 'https://github.com/vedanggggg/cyclone-forecasting'
REPO_CLONE_DIR = Path('/kaggle/working/forecast-diffmodels')
IMAGEN_PYTORCH_DIR = Path('/kaggle/working/imagen-pytorch')

assert DATALOADER_SRC.exists(), f'Missing: {DATALOADER_SRC}'
assert TRAIN_SPLIT_PKL.exists(), f'Missing: {TRAIN_SPLIT_PKL}'
assert TEST_SPLIT_PKL.exists(), f'Missing: {TEST_SPLIT_PKL}'

print('Input validation passed.')
print('Cyclone dat files:', len(list(DATALOADER_SRC.glob('*.dat'))))


## Step 1: Clone Repositories

Run this once.
Expected result:
- `forecast-diffmodels` exists under `/kaggle/working`
- `imagen-pytorch` exists under `/kaggle/working`


In [ ]:
import subprocess

if not REPO_CLONE_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_CLONE_DIR)], check=True)
else:
    print('Repo already exists:', REPO_CLONE_DIR)

if not IMAGEN_PYTORCH_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/lucidrains/imagen-pytorch.git', str(IMAGEN_PYTORCH_DIR)], check=True)
else:
    print('imagen-pytorch already exists:', IMAGEN_PYTORCH_DIR)


## Step 2: Detect Real Project Root and Normalize Folder Names

The public repo may contain an extra nested folder and/or `imagen ` with trailing space.
Run this cell.
Expected result:
- `PROJECT_ROOT` printed
- both `dataproc/utils.py` and `imagen/helpers.py` found


In [ ]:
import os

# Fix trailing-space imagen folder if present anywhere inside clone
for p in REPO_CLONE_DIR.rglob('imagen '):
    target = p.parent / 'imagen'
    if not target.exists():
        os.rename(p, target)
        print('Renamed:', p, '->', target)

candidates = []
for utils_path in REPO_CLONE_DIR.rglob('dataproc/utils.py'):
    root = utils_path.parent.parent
    if (root / 'imagen' / 'helpers.py').exists():
        candidates.append(root)

assert candidates, 'Could not find project root containing both dataproc/utils.py and imagen/helpers.py'
PROJECT_ROOT = sorted(candidates, key=lambda x: len(str(x)))[0]

IMAGEN_DIR = PROJECT_ROOT / 'imagen'
DATAPROC_DIR = PROJECT_ROOT / 'dataproc'
STAGE1_DIR = IMAGEN_DIR / '64_FC'

assert (DATAPROC_DIR / 'utils.py').exists()
assert (IMAGEN_DIR / 'helpers.py').exists()
assert (STAGE1_DIR / 'train64.py').exists()

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATAPROC_DIR =', DATAPROC_DIR)
print('IMAGEN_DIR   =', IMAGEN_DIR)
print('STAGE1_DIR   =', STAGE1_DIR)


## Step 3: Install Required Dependencies

Run this cell.
Notes:
- Uses minimal install needed for Stage-1 instead of full pinned HPC environment.
- If installation fails, stop and fix before next step.


In [ ]:
import sys
import subprocess

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(IMAGEN_PYTORCH_DIR)], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'einops', 'pixelmatch', 'torchmetrics', 'lpips',
    'xarray', 'satpy', 'fsspec', 'pyproj', 'scipy', 'scikit-image',
    'openpyxl', 'dill', 'pandas', 'h5netcdf', 'xarray-datatree',
    'tensorboard', 'opencv-python'
], check=True)

import imagen_pytorch
print('imagen_pytorch import OK:', imagen_pytorch.__file__)


## Step 4: Create Author-Expected `/rds/...` Layout

Original scripts use hardcoded `/rds/...` roots.
Run this cell to create those folders and link code there.


In [ ]:
import shutil

RDS_HOME = Path('/rds/general/user/zr523/home/researchProject')
RDS_DATA = Path('/rds/general/ephemeral/user/zr523/ephemeral')

(RDS_HOME).mkdir(parents=True, exist_ok=True)
(RDS_DATA / 'satellite' / 'dataloader' / '64_FC').mkdir(parents=True, exist_ok=True)

# Link project root to exact path expected by helpers.py
link = RDS_HOME / 'forecast-diffmodels'
if link.is_symlink():
    link.unlink()
elif link.exists():
    shutil.rmtree(link)
os.symlink(str(PROJECT_ROOT), str(link))

print('Symlink:', link, '->', os.readlink(link))


## Step 5: Copy and Rename Cyclone `.dat` Files to Author Format

Author loader expects files as `region_name.dat`.
Your dataset files are `name_region.dat`.
Run this cell.


In [ ]:
import shutil

DATALOADER_DST = Path('/rds/general/ephemeral/user/zr523/ephemeral/satellite/dataloader/64_FC')
DATALOADER_DST.mkdir(parents=True, exist_ok=True)

# Clean destination to avoid stale duplicates
for old in DATALOADER_DST.glob('*.dat'):
    old.unlink()

for f in sorted(DATALOADER_SRC.glob('*.dat')):
    stem = f.stem
    assert '_' in stem, f'Unexpected filename (missing underscore): {f.name}'
    name, region = stem.rsplit('_', 1)
    out = DATALOADER_DST / f'{region}_{name}.dat'
    shutil.copy2(f, out)

files = sorted([x.name for x in DATALOADER_DST.glob('*.dat')])
print('Copied files:', len(files))
print('First 5:', files[:5])


## Step 6: Reconstruct `test_set.pkl` from Provided Train/Test Loader Pickles

Original code splits by cyclone names from `dataproc/test_set.pkl`.
This cell reconstructs that mapping so author code reproduces your provided split.

Expected:
- `test_set.pkl` written to `/rds/general/user/zr523/home/researchProject/forecast-diffmodels/dataproc/test_set.pkl`


In [ ]:
import pickle
from itertools import combinations

# Placeholder classes required for unpickling objects saved from notebooks
class CycloneDataLoader: pass
class ModelDataLoader: pass

with open(TEST_SPLIT_PKL, 'rb') as f:
    test_loader = pickle.load(f)
with open(TRAIN_SPLIT_PKL, 'rb') as f:
    train_loader = pickle.load(f)

target_test = int(test_loader.img_o.shape[0])
target_train = int(train_loader.img_o.shape[0])

items = []
for fp in sorted(DATALOADER_DST.glob('*.dat')):
    region, name = fp.stem.split('_', 1)
    with open(fp, 'rb') as f:
        obj = pickle.load(f)
    items.append((region, name, int(obj.img_64.shape[0])))

total = sum(x[2] for x in items)
idxs = list(range(len(items)))

matches = []
for r in range(1, len(items) + 1):
    for comb in combinations(idxs, r):
        test_count = sum(items[i][2] for i in comb)
        train_count = total - test_count
        if test_count == target_test and train_count == target_train:
            matches.append(comb)

assert matches, 'No cyclone subset matched provided train/test sample counts.'

# If multiple matches exist, use smallest set and print alternatives.
matches = sorted(matches, key=len)
chosen = matches[0]

split = {'nio': [], 'aus': [], 'wpo': [], 'wio': [], 'use': [], 'usw': []}
for i in chosen:
    region, name, _ = items[i]
    split[region].append(name)

if len(matches) > 1:
    print('WARNING: multiple valid test subsets found. Using smallest one by default.')
    print('Num matches:', len(matches))

test_set_path = Path('/rds/general/user/zr523/home/researchProject/forecast-diffmodels/dataproc/test_set.pkl')
with open(test_set_path, 'wb') as f:
    pickle.dump(split, f)

print('test_set.pkl written:', test_set_path)
print('test split mapping:', split)
print('target train/test:', target_train, target_test)


## Step 7: Validate Original Imports and Dataloader Construction

Run this cell before training.
Expected:
- `helpers import OK`
- non-zero train/test batch counts


In [ ]:
import sys

# Force import paths explicitly
if str(DATAPROC_DIR) not in sys.path:
    sys.path.insert(0, str(DATAPROC_DIR))
if str(IMAGEN_DIR) not in sys.path:
    sys.path.insert(0, str(IMAGEN_DIR))
if str(IMAGEN_PYTORCH_DIR) not in sys.path:
    sys.path.insert(0, str(IMAGEN_PYTORCH_DIR))

import utils
from helpers import get_satellite_data
print('helpers import OK')

class Args: pass
args = Args()
args.batch_size = 1
args.o_size = 64
args.n_size = 128
args.dataset_path = str(DATALOADER_DST)
args.datalimit = False
args.mode = 'fc'
args.lr = 3e-4
args.augment = False

train_dl, test_dl = get_satellite_data(args, 'vid')
print('train batches:', len(train_dl))
print('test batches :', len(test_dl))


## Step 8: Run Stage-1 Training (Original Script)

This executes original `train64.py`.
It may take many hours.


In [ ]:
import os
import subprocess

env = os.environ.copy()
env['PYTHONPATH'] = f"{DATAPROC_DIR}:{IMAGEN_DIR}:{IMAGEN_PYTORCH_DIR}:{env.get('PYTHONPATH','')}"

subprocess.run(
    ['python', 'train64.py', '-mode', 'execute', '-run_name', 'v_FC_dim64', '-epochs', '400'],
    cwd=str(STAGE1_DIR),
    env=env,
    check=True
)


## Step 9: Run Evaluation (Original Script)

Run after training finishes.


In [ ]:
import os
import subprocess

env = os.environ.copy()
env['PYTHONPATH'] = f"{DATAPROC_DIR}:{IMAGEN_DIR}:{IMAGEN_PYTORCH_DIR}:{env.get('PYTHONPATH','')}"

subprocess.run(
    ['python', 'v_t02-sampling-and-evaluation.py', '-run_name', 'v_FC_dim64'],
    cwd=str(STAGE1_DIR),
    env=env,
    check=True
)


## Step 10 (Optional): Test-Set Metrics at Specific Epoch

Use this only after checkpoints are available.
- If you want exact paper-style run, set `BEST_EPOCH = 395`.
- Otherwise use latest available checkpoint.


In [ ]:
import glob
import re
import os
import subprocess
from pathlib import Path

ckpt_dir = Path('/rds/general/user/zr523/home/researchProject/models/v_FC_dim64/models/v_FC_dim64')
ckpts = sorted(glob.glob(str(ckpt_dir / 'ckpt_1_*.pt')))
assert ckpts, f'No checkpoints found in {ckpt_dir}'

latest_epoch = max(int(re.search(r'_(\d{3})\.pt$', x).group(1)) for x in ckpts)
BEST_EPOCH = latest_epoch  # or set manually to 395
print('Using BEST_EPOCH =', BEST_EPOCH)

env = os.environ.copy()
env['PYTHONPATH'] = f"{DATAPROC_DIR}:{IMAGEN_DIR}:{IMAGEN_PYTORCH_DIR}:{env.get('PYTHONPATH','')}"

subprocess.run(
    ['python', 'test64.py', '-run_name', 'v_FC_dim64', '-best_epoch', str(BEST_EPOCH)],
    cwd=str(STAGE1_DIR),
    env=env,
    check=True
)
